# 04 — Implement normalized difference

Calculate NDWI from ordered raster inputs and reject incompatible grids.

Run this lab in the **dev-platform** workspace on Minikube or Kubernetes, using the `eoepca/pde-code-server:amd64` image. Select its preinstalled **Bash** kernel and use **Shift+Enter**, reading each explanation before executing its cell. No terminal or kernel installation is needed. Stop at an unexpected error and fix it before continuing. The course environment is provisioned by workspace startup.

The kernel keeps environment variables and the working directory between cells. The first cell finds the course checkout and activates its locked environment. Command groups run in a subshell with `set -euo pipefail` so a failing command stops that cell without killing the kernel. Scratch results stay in `.notebook-runs/04-implement-norm-diff/`; rerunning a cell replaces only its lab outputs.

[Lesson](../modules/04-implement-norm-diff/lesson.md) · [Exercise](../modules/04-implement-norm-diff/exercise.md)


## Prepare the notebook workspace

Run this cell after every kernel restart. It clears inherited `CWLTOOL_OPTIONS` so a Coder workspace configured for Podman can still run the explicitly local exercises. It does not change your cluster or publish anything.


In [ ]:
lab_start_dir="$PWD"
while [[ ! -f reference/waterbodies.cwl || ! -f pyproject.toml ]]; do
    if [[ "$PWD" == / ]]; then
        cd "$lab_start_dir"
        break
    fi
    cd ..
done
export WORKSPACE="$PWD"
if [[ -x "$WORKSPACE/.venv/bin/python" && -f "$WORKSPACE/reference/waterbodies.cwl" ]]; then
    source "$WORKSPACE/.venv/bin/activate"
    export PYTHONPATH="$WORKSPACE${PYTHONPATH:+:$PYTHONPATH}"
    export CWLTOOL_OPTIONS=
    export MPLBACKEND=Agg
    export LAB_DIR="$WORKSPACE/.notebook-runs/04-implement-norm-diff"
    mkdir -p "$LAB_DIR"
    cd "$LAB_DIR"
    python --version
    waterbodies --help
else
    printf 'Open this notebook inside the course checkout. Ask the instructor to check workspace environment provisioning.\n' >&2
    false
fi


## Create matching inputs

This lab creates its own scene and crops, so it can be restarted independently of module 03. Predict why the raster order is part of the contract.


In [ ]:
(
set -euo pipefail
python - <<'PYTHON'
from pathlib import Path
from scripts.fixtures import create_fixture
create_fixture(Path('source'))
print('Created an 8 x 8 STAC scene with green and NIR assets under source/')
PYTHON

waterbodies crop --input-item source --aoi "1,1,7,7" --epsg EPSG:4326 --band green
waterbodies crop --input-item source --aoi "1,1,7,7" --epsg EPSG:4326 --band nir
)


## Calculate and inspect NDWI

For the left half `(100-300)/(100+300)=-0.5`; for the right half the value is `+0.5`. Both inputs must describe the same grid.


In [ ]:
(
set -euo pipefail
waterbodies norm-diff --rasters crop_green.tif --rasters crop_nir.tif

python - <<'PYTHON'
import numpy as np
import rasterio
with rasterio.open('norm_diff.tif') as raster:
    pixels = raster.read(1)
    print(raster.dtypes, raster.shape, np.unique(pixels))
    assert raster.dtypes == ('float32',)
    np.testing.assert_array_equal(pixels[:, :3], -0.5)
    np.testing.assert_array_equal(pixels[:, 3:], 0.5)
np.save('original.npy', pixels)
PYTHON
)


## Reverse the ordered inputs

Predict the sign change. Save and compare the observations, then restore the default order for downstream processing.


In [ ]:
(
set -euo pipefail
waterbodies norm-diff --rasters crop_nir.tif --rasters crop_green.tif
python - <<'PYTHON'
import numpy as np
import rasterio
with rasterio.open('norm_diff.tif') as raster:
    reversed_pixels = raster.read(1)
np.testing.assert_array_equal(reversed_pixels, -np.load('original.npy'))
print('Reversed values:', np.unique(reversed_pixels))
PYTHON
waterbodies norm-diff --rasters crop_green.tif --rasters crop_nir.tif
)


## Reject mismatched grids

Create a smaller NIR crop and keep the original green crop. An array pair with unequal grids cannot describe per-pixel correspondence; expected rejection is handled explicitly.


In [ ]:
(
set -euo pipefail
waterbodies crop --input-item source --aoi "2,2,7,7" --epsg EPSG:4326 --band nir
if waterbodies norm-diff --rasters crop_green.tif --rasters crop_nir.tif > mismatch.log 2>&1; then
    printf 'Unexpected success for mismatched grids.\n' >&2
    exit 1
fi
cat mismatch.log
waterbodies crop --input-item source --aoi "1,1,7,7" --epsg EPSG:4326 --band nir
waterbodies norm-diff --rasters crop_green.tif --rasters crop_nir.tif

python -m pytest "$WORKSPACE/tests/tools/test_processing.py" -k "norm_diff or complete"
)


## Record your findings

**Expected outcome:** The float32 NDWI contains −0.5/+0.5, swapping bands reverses its sign, and unequal grids are rejected.

Add your observations to a Markdown cell: what you predicted, what the command produced, and which contract declaration explains it. Restart the kernel and run all cells with the default settings to check reproducibility.

Compare [the solution](../modules/04-implement-norm-diff/solution.md) after the activity. Return to [the lab index](README.md).
